# Unsupervised Bipartite AML GNN Tutorial

This notebook demonstrates how to use the updated `bipartite_aml_gnn.py` script for **unsupervised anomaly detection**. 

### Workflow:
1. **Graph Construction**: Build a bipartite network of Customers and Bank Accounts.
2. **Self-Supervised Training**: Train the GNN using **Link Prediction** (Negative Sampling) to learn structural embeddings without labels.
3. **Anomaly Detection**: Use **Isolation Forest** on the learned embeddings to identify high-risk outliers.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# Add the scripts directory to the system path to import the module
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'scripts')))

from bipartite_aml_gnn import prepare_hetero_data, UnifiedAMLGNN, train_unsupervised, inference_anomaly_detection

## 1. Load or Generate Data

We generate mock data simulating transactions. Notice we no longer require a `label` column.

In [ ]:
# Generate Mock Data
num_tx = 1000
mock_data = {
    'transaction_id': range(num_tx),
    'customer_name': [f'Cust_{np.random.randint(0, 100)}' for _ in range(num_tx)],
    'bank_account_id': [f'Acct_{np.random.randint(0, 50)}' for _ in range(num_tx)],
    'amount': np.random.rand(num_tx) * 10000,
    'hour_of_day': np.random.randint(0, 24, num_tx),
    # Customer Features
    'cust_risk_score': np.random.rand(num_tx),
    'cust_tenure': np.random.randint(1, 365, num_tx),
    # Bank Features
    'bank_volume': np.random.rand(num_tx) * 100000,
    'bank_flags': np.random.randint(0, 2, num_tx),
}
df = pd.DataFrame(mock_data)

print(f"Generated {len(df)} transactions.")
df.head()

## 2. Configure Columns

Define IDs and feature columns.

In [ ]:
CUSTOMER_ID = 'customer_name'
BANK_ID = 'bank_account_id'

CUSTOMER_FEATURES = ['cust_risk_score', 'cust_tenure']
BANK_FEATURES = ['bank_volume', 'bank_flags']
EDGE_FEATURES = ['amount', 'hour_of_day']

## 3. Prepare Graph Data

In [ ]:
hetero_data, cust_mapping = prepare_hetero_data(
    df, CUSTOMER_ID, BANK_ID, 
    CUSTOMER_FEATURES, BANK_FEATURES, EDGE_FEATURES
)

print(hetero_data)

## 4. Unsupervised Training

The model learns to predict if an edge should exist, forcing it to create meaningful embeddings for both nodes.

In [ ]:
MODEL_CHOICE = 'sage'

model = UnifiedAMLGNN(
    hidden_channels=64, 
    out_channels=64, # Dimension of the learned embeddings
    num_layers=2,
    architecture=MODEL_CHOICE
)

print("Starting Unsupervised Training (Link Prediction)...")
model = train_unsupervised(model, hetero_data, epochs=10, lr=0.01)

## 5. Anomaly Detection Results

Isolation Forest identifies customers whose GNN embeddings are outliers compared to the general population.

In [ ]:
results = inference_anomaly_detection(model, hetero_data)

# Map back to original IDs
inv_cust_map = {v: k for k, v in cust_mapping.items()}
results['customer_id'] = results['cust_idx'].map(inv_cust_map)

print("Top 10 Most Anomalous Customers:")
results.sort_values('risk_score', ascending=False).head(10)